# Load from Pickle

In [1]:
import pandas as pd
import numpy as np

In [2]:
import os

print(os.getcwd()) #zeigt Arbeitsverzeichnis des Kernels
print(os.listdir()) #zeigt Dateien, die sich darin befinden


d:\iCloudDrive\Dokumente\Weiterbildung\Data Analyst\Abschlussprojekt\olist-marketplace-analysis\olist-marketplace-analysis\scripts\silver
['.gitkeep', 'aws_lambda_clean_transform_tables.py', 'silver_clean_customers_orders_order_reviews_en.py', 'silver_clean_customers_orders_order_review_en.ipynb', 'silver_clean_geo_location.order_items.product_category_names.py', 'silver_clean_order_payments_products_sellers.ipynb', 'silver_clean_order_payments_products_sellers.py', 'silver_create_athena_tables.sql', 'silver_geo_location.order_items.product_category_names.ipynb', 'silver_geo_location.order_items.product_category_names.py']


In [3]:

last_layer = "bronze"
current_layer = "silver"
source_path = f"../{last_layer}/"
source_type = ".pkl"


bronze_tables_dict = pd.read_pickle(f"{source_path}{last_layer}_tables{source_type}")

print(f"Pickle Tables ({", ".join(bronze_tables_dict.keys())}) successsfully loaded into {current_layer}-Layer!")

Pickle Tables (bronze.csv_order_payments.csv, bronze.csv_products.csv, bronze.csv_sellers.csv) successsfully loaded into silver-Layer!


In [4]:
display(bronze_tables_dict["bronze.csv_order_payments.csv"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [5]:
df_order_payments = bronze_tables_dict["bronze.csv_order_payments.csv"]
df_products = bronze_tables_dict["bronze.csv_products.csv"]
df_sellers = bronze_tables_dict["bronze.csv_sellers.csv"]


# Start Cleaning order_payments

In [6]:
display(df_order_payments.head())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Explain the columns

**order_id**: 
unique identifier of an order

**payment_sequential**: 
a customer may pay an order with more than one payment method. If he does so, a sequence will be created to accommodate all payments

**payment_type**: 
method of payment chosen by the customer

**payment_installments**: 
number of installments chosen by the customer

**payment_value**: 
transaction value


## Convert columns

In [7]:
numeric_columns = ["payment_sequential", "payment_installments", "payment_value"]
string_columns = ["order_id", "payment_type"]

for col in numeric_columns:

    try:
        df_order_payments[col] = pd.to_numeric(df_order_payments[col], errors="raise")
        print(f"{col} successfully converted into {df_order_payments[col].dtype}")
        
    except Execution as e:
        print(f"Error in converting {col} into numeric")
        print(f"Error details: {e}")
        sys.exit(1)
        

for col in string_columns:

    try:
        df_order_payments[col] = df_order_payments[col].astype("str")
        print(f"{col} successfully converted into {df_order_payments[col].dtype}")
        
    except Execution as e:
        print(f"Error in converting {col} into string")
        print(f"Error details: {e}")
        sys.exit(1)

payment_sequential successfully converted into int64
payment_installments successfully converted into int64
payment_value successfully converted into float64
order_id successfully converted into str
payment_type successfully converted into str


## Check Data Quality

In [8]:
print(df_order_payments.info())

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB
None


### payment_type

In [9]:
print(f"""
payment_type has the following values: 
{df_order_payments["payment_type"].value_counts()}
""")


payment_type has the following values: 
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64



In [10]:
#dropping rows with 'not_defined' payment_type

payment_type_is_not_defined_count = df_order_payments[df_order_payments.payment_type == "not_defined"].shape[0]

df_order_payments = df_order_payments[df_order_payments.payment_type != "not_defined"]

print(f"{payment_type_is_not_defined_count} rows with payment_type 'not_defined' were dropped")

3 rows with payment_type 'not_defined' were dropped


### payment_installments

In [11]:
print(f"""
payment_installments has the following statistic values:
{df_order_payments["payment_installments"].describe()}
""")


payment_installments has the following statistic values:
count    103883.000000
mean          2.853402
std           2.687071
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64



### payment_sequential

In [12]:
print(f"""
payment_sequential has the following statistic values:
{df_order_payments["payment_sequential"].describe()}
""")


payment_sequential has the following statistic values:
count    103883.000000
mean          1.092681
std           0.706594
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          29.000000
Name: payment_sequential, dtype: float64



### payment_value

In [13]:
print(f"""
payment_value has the following statistic values:
{df_order_payments["payment_value"].describe()}
""")


payment_value has the following statistic values:
count    103883.000000
mean        154.104831
std         217.495628
min           0.000000
25%          56.800000
50%         100.000000
75%         171.840000
max       13664.080000
Name: payment_value, dtype: float64



In [14]:
#dropping rows with '0.0' payment_value

payment_value_is_zero_count = df_order_payments[df_order_payments.payment_value == 0].shape[0]

df_order_payments = df_order_payments[df_order_payments.payment_value != 0]

print(f"{payment_value_is_zero_count} rows with payment_value '0.0' were dropped")

6 rows with payment_value '0.0' were dropped


### Built a unique key

In [15]:
#starts at 1

df_order_payments["id_order_payments"] = np.arange(1, df_order_payments.shape[0] + 1)

In [16]:
sorted_columns = df_order_payments.columns.sort_values()
df_order_payments = df_order_payments[sorted_columns]
#df_order_payments

# Start Cleaning products

In [17]:
display(df_products.head())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


### Explain the columns

**product_id**: 
unique product identifier

**product_category_name**: 
root category of product, in Portuguese

**product_name_lenght**: 
number of characters extracted from the product name


**product_description_lenght**: 
number of characters extracted from the product description


**product_photos_qty**: 
number of product published photos


**product_weight_g**:
product weight measured in grams


**product_length_cm**:
product length measured in centimeters


**product_height_cm**:
product height measured in centimeters


**product_width_cm**:
product width measured in centimeters

## Convert columns

In [18]:
numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
    ]

string_columns = [
    "product_id",
    "product_category_name"
]

for col in numeric_columns:

    try:
        df_products[col] = pd.to_numeric(df_products[col], errors="raise")
        print(f"{col} successfully converted into {df_products[col].dtype}")
        
    except Execution as e:
        print(f"Error in converting {col} into numeric")
        print(f"Error details: {e}")
        sys.exit(1) #exit script to avoid errors


for col in string_columns:

    try:
        df_products[col] = df_products[col].astype("str")
        print(f"{col} successfully converted into {df_products[col].dtype}")
        
    except Execution as e:
        print(f"Error in converting {col} into string")
        print(f"Error details: {e}")
        sys.exit(1)

product_name_lenght successfully converted into float64
product_description_lenght successfully converted into float64
product_photos_qty successfully converted into float64
product_weight_g successfully converted into float64
product_length_cm successfully converted into float64
product_height_cm successfully converted into float64
product_width_cm successfully converted into float64
product_id successfully converted into str
product_category_name successfully converted into str


## Check Data Quality

In [19]:
print(df_products.info())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB
None


### product_id

In [20]:
#check uniqueness of product_id

if (df_products.dropna(subset= "product_id").shape[0] == df_products.shape[0]) & (df_products.product_id.nunique() == df_products.product_id.shape[0]):
    print("Every row has a product_id and they are unique")
    product_id_uniqueness = True

elif df_products.dropna(subset= "product_id").shape[0] < df_products.shape[0]:
    
    print(f"""
    There are {df_products.shape[0] - df_products.dropna(subset= "product_id").shape[0]} rows without a product_id
    """)

else:
    print(f"""
    There are duplicates in product_id:
    {display(df_products[df_products.duplicated(subset="product_id", keep=False)].sort_values(by="product_id"))}
    """)
    sys.exit(1) #exit script to avoid errors


Every row has a product_id and they are unique


In [21]:
#Are there rows, where all data columns (without product_id) are null?

empty_data_rows_count = df_products[df_products.drop("product_id", axis=1).isnull().all(axis=1)].shape[0]
empty_data_rows_with_product_id_count = df_products[df_products.drop("product_id", axis=1).isnull().all(axis=1)].dropna(subset= "product_id").shape[0]
empty_rows_without_product_id_count = empty_data_rows_count - empty_data_rows_with_product_id_count

data_columns = df_products.columns.drop("product_id")

if empty_data_rows_count > 0:
    
    print(f"{empty_data_rows_count} products don't have data in data columns")
    print(f"{empty_data_rows_with_product_id_count} products have only a product_id and don't have data")
    print(f"{empty_rows_without_product_id_count} products don't have any data")

    #drop rows without data in data columns

    df_products = df_products.dropna(subset=data_columns, how="all")
    empty_data_rows_after_drop_count = df_products[df_products.drop("product_id", axis=1).isnull().all(axis=1)].shape[0]

    if empty_data_rows_count - empty_data_rows_after_drop_count == empty_data_rows_count:
        print(f"Every product without data in data columns ({", ".join(data_columns)}) were dropped.")
    else:
        print("There was a problem by dropping empty data rows")

else:
    print("There are no products without data")

1 products don't have data in data columns
1 products have only a product_id and don't have data
0 products don't have any data
Every product without data in data columns (product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm) were dropped.


### product_category_name

In [22]:
#check, if products have multiple categories

category_seperator_list = [",", ";", " "]

for sep in category_seperator_list:

    if df_products[(df_products.product_category_name != df_products.product_category_name.str.replace(sep,"")) & (df_products.product_category_name.isnull() == False)].shape[0] == 0:
        print(f"There are no products, which have multiple categories seperated with '{sep}'")

    else:
        print(f"There are products, which have multiple categories seperated with '{sep}':\n")
        print(df_products[(df_products.product_category_name != df_products.product_category_name.str.replace(sep,"")) & (df_products.product_category_name.isnull() == False)][["product_id", "product_category_name"]])

There are no products, which have multiple categories seperated with ','
There are no products, which have multiple categories seperated with ';'
There are no products, which have multiple categories seperated with ' '


In [23]:
print(f"product_category_name has {df_products.product_category_name.nunique()} unique categories")
print(f"In product_category_name {df_products.product_category_name.isnull().sum()} of {df_products.shape[0]} products don't have any category (NULL values)")

product_category_name has 73 unique categories
In product_category_name 609 of 32950 products don't have any category (NULL values)


### product_description_lenght

In [24]:
print(f"product_description_lenght is from {df_products.product_description_lenght.min():.0f} up to {df_products.product_description_lenght.max():.0f} characters and the mean description lenght is {df_products.product_description_lenght.mean():.0f}")
print(f"In product_description_lenght {df_products.product_description_lenght.isnull().sum()} of {df_products.shape[0]} products don't have a description (NULL values)")

product_description_lenght is from 4 up to 3992 characters and the mean description lenght is 771
In product_description_lenght 609 of 32950 products don't have a description (NULL values)


### product_photos_qty

In [25]:
print(f"product_photos_qty is from {df_products.product_photos_qty.min():.0f} up to {df_products.product_photos_qty.max():.0f} photos per product and the mean photo quantity is {df_products.product_photos_qty.mean():.0f}")
print(f"In product_photos_qty {df_products.product_photos_qty.isnull().sum()} of {df_products.shape[0]} products don't have any photo (NULL values)")

product_photos_qty is from 1 up to 20 photos per product and the mean photo quantity is 2
In product_photos_qty 609 of 32950 products don't have any photo (NULL values)


In [26]:
#Are there products without a category, description and photos?

data_columns = ["product_category_name", "product_description_lenght", "product_photos_qty"]

print(df_products[df_products[data_columns].isnull().all(axis=1)].head())

empty_data_rows_count = df_products[df_products[data_columns].isnull().all(axis=1)].shape[0]

if empty_data_rows_count > 0:
    
    print(f"{empty_data_rows_count} of {df_products.shape[0]} rows don't have data in any data column ({", ".join(data_columns)})")
    
else:
    print(f"There are no products without data in {", ".join(data_columns)}")

                           product_id product_category_name  \
105  a41e356c76fab66334f36de622ecbd3a                   NaN   
128  d8dee61c2034d6d075997acef1870e9b                   NaN   
145  56139431d72cd51f19eb9f7dae4d1617                   NaN   
154  46b48281eb6d663ced748f324108c733                   NaN   
197  5fb61f482620cb672f5e586bb132eae9                   NaN   

     product_name_lenght  product_description_lenght  product_photos_qty  \
105                  NaN                         NaN                 NaN   
128                  NaN                         NaN                 NaN   
145                  NaN                         NaN                 NaN   
154                  NaN                         NaN                 NaN   
197                  NaN                         NaN                 NaN   

     product_weight_g  product_length_cm  product_height_cm  product_width_cm  
105             650.0               17.0               14.0              12.0  
128 

###  product_weight_g, product_length_cm, product_height_cm, product_width_cm   

In [27]:
#product_weight_g

print(f"product_weight_g is from {df_products.product_weight_g.min():.0f} up to {df_products.product_weight_g.max():.0f} grams per product and the mean weight is {df_products.product_weight_g.mean():.0f}")
print(f"In product_weight_g {df_products.product_weight_g.isnull().sum()} of {df_products.shape[0]} products don't have any weight")

product_weight_g is from 0 up to 40425 grams per product and the mean weight is 2276
In product_weight_g 1 of 32950 products don't have any weight


In [28]:
#product_length_cm

print(f"product_length_cm is from {df_products.product_length_cm.min():.0f} up to {df_products.product_length_cm.max():.0f} centimetres per product and the mean lenghth is {df_products.product_length_cm.mean():.0f}")
print(f"In product_length_cm {df_products.product_length_cm.isnull().sum()} of {df_products.shape[0]} products don't have any length")

product_length_cm is from 7 up to 105 centimetres per product and the mean lenghth is 31
In product_length_cm 1 of 32950 products don't have any length


In [29]:
#product_height_cm

print(f"product_height_cm is from {df_products.product_height_cm.min():.0f} up to {df_products.product_height_cm.max():.0f} centimetres per product and the mean height is {df_products.product_height_cm.mean():.0f}")
print(f"In product_height_cm {df_products.product_height_cm.isnull().sum()} of {df_products.shape[0]} products don't have any height")

product_height_cm is from 2 up to 105 centimetres per product and the mean height is 17
In product_height_cm 1 of 32950 products don't have any height


In [30]:
#product_width_cm

print(f"product_width_cm is from {df_products.product_width_cm.min():.0f} up to {df_products.product_width_cm.max():.0f} centimetres per product and the mean width is {df_products.product_width_cm.mean():.0f}")
print(f"In product_width_cm {df_products.product_width_cm.isnull().sum()} of {df_products.shape[0]} products don't have any width")

product_width_cm is from 6 up to 118 centimetres per product and the mean width is 23
In product_width_cm 1 of 32950 products don't have any width


In [31]:
#Are there products without data in size columns?

data_columns = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(df_products[df_products[data_columns].isnull().all(axis=1)].head())
empty_data_rows_count = df_products[df_products[data_columns].isnull().all(axis=1)].shape[0]

if empty_data_rows_count > 0:
    
    print(f"{empty_data_rows_count} of {df_products.shape[0]} rows don't have data in any data column ({", ".join(data_columns)})")

else:
    print(f"There no products without data in all size columns ({", ".join(data_columns)})")
    

                            product_id product_category_name  \
8578  09ff539a621711667c43eba6a3bd8466                 bebes   

      product_name_lenght  product_description_lenght  product_photos_qty  \
8578                 60.0                       865.0                 3.0   

      product_weight_g  product_length_cm  product_height_cm  product_width_cm  
8578               NaN                NaN                NaN               NaN  
1 of 32950 rows don't have data in any data column (product_weight_g, product_length_cm, product_height_cm, product_width_cm)


### Built a shorter unique key

In [32]:
if product_id_uniqueness == True:
    
    #starts at 1

    df_products["id_products"] = np.arange(1, df_products.shape[0] + 1)
    

In [33]:
sorted_columns = df_products.columns.sort_values()
df_products = df_products[sorted_columns]
#df_products

# Start Cleaning sellers

In [34]:
display(df_sellers.head())

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Explain the columns

**seller_id**: 
seller unique identifier

**seller_zip_code_prefix**: 
first 5 digits of seller zip code

**seller_city**: 
seller city name

**seller_state**: 
seller state

## Convert columns

In [35]:

string_columns = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]

for col in string_columns:

    try:
        df_sellers[col] = df_sellers[col].astype("str")
        print(f"{col} successfully converted into {df_sellers[col].dtype}")
        
    except Execution as e:
        print(f"Error in converting {col} into string")
        print(f"Error details: {e}")
        sys.exit(1)

seller_id successfully converted into str
seller_zip_code_prefix successfully converted into str
seller_city successfully converted into str
seller_state successfully converted into str


## Check Data Quality

In [36]:
print(df_sellers.info())

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 245.5 KB
None


In [37]:
#check uniqueness of seller_id

if (df_sellers.dropna(subset= "seller_id").shape[0] == df_sellers.shape[0]) & (df_sellers.seller_id.nunique() == df_sellers.seller_id.shape[0]):
    print("Every row has a seller_id and they are unique")
    seller_id_uniqueness = True

elif df_sellers.dropna(subset= "seller_id").shape[0] < df_sellers.shape[0]:
    
    print(f"""
    There are {df_sellers.shape[0] - df_sellers.dropna(subset= "seller_id").shape[0]} rows without a seller_id
    """)

else:
    print(f"""
    There are duplicates in seller_id:
    {display(df_sellers[df_sellers.duplicated(subset="seller_id", keep=False)].sort_values(by="seller_id"))}
    """)
    sys.exit(1) #exit script to avoid errors


Every row has a seller_id and they are unique


In [38]:
#Are there rows, where all data columns (without seller_id) are null?

empty_data_rows_count = df_sellers[df_sellers.drop("seller_id", axis=1).isnull().all(axis=1)].shape[0]
empty_data_rows_with_seller_id_count = df_sellers[df_sellers.drop("seller_id", axis=1).isnull().all(axis=1)].dropna(subset= "seller_id").shape[0]
empty_rows_without_seller_id_count = empty_data_rows_count - empty_data_rows_with_seller_id_count

data_columns = df_sellers.columns.drop("seller_id")

if empty_data_rows_count > 0:
    
    print(f"{empty_data_rows_count} rows don't have data in data columns")
    print(f"{empty_data_rows_with_seller_id_count} rows have only a seller_id and don't have data")
    print(f"{empty_rows_without_seller_id_count} rows don't have any data")

    #drop rows without data in data columns

    df_sellers = df_sellers.dropna(subset=data_columns, how="all")
    empty_data_rows_after_drop_count = df_sellers[df_sellers.drop("seller_id", axis=1).isnull().all(axis=1)].shape[0]

    if empty_data_rows_count - empty_data_rows_after_drop_count == empty_data_rows_count:
        print(f"Every row without data in data columns ({", ".join(data_columns)}) were dropped.")
    else:
        print("There was a problem by dropping empty data rows")

else:
    print(f"There are no seller_id without data in data columns ({", ".join(data_columns)})")


There are no seller_id without data in data columns (seller_zip_code_prefix, seller_city, seller_state)


### seller_zip_code_prefix

In [39]:
df_sellers.seller_zip_code_prefix.value_counts().describe()


count    2246.000000
mean        1.378005
std         1.294510
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        49.000000
Name: count, dtype: float64

In [40]:
print(f"""
There are zip code prefix regions where only {df_sellers.seller_zip_code_prefix.value_counts().min():.0f} sellers are located
and regions there are {df_sellers.seller_zip_code_prefix.value_counts().max():.0f} sellers located.
In average there are {df_sellers.seller_zip_code_prefix.value_counts().mean():.0f} seller per zip code prefix region
""")


There are zip code prefix regions where only 1 sellers are located
and regions there are 49 sellers located.
In average there are 1 seller per zip code prefix region



### seller_city

In [41]:
#pip install unidecode

In [42]:
from unidecode import unidecode

In [43]:
cities_with_special_chars = df_sellers[df_sellers["seller_city"].str.lower().apply(unidecode) != df_sellers["seller_city"]]

if cities_with_special_chars.shape[0] > 0:

    print(f"""
    {", ".join(cities_with_special_chars.seller_city)} are cities with special characters as letters.
    Convert to english character set... 
    """)
    df_sellers["seller_city"] = df_sellers["seller_city"].apply(unidecode)
    cities_with_special_chars = df_sellers[df_sellers["seller_city"].str.lower().apply(unidecode) != df_sellers["seller_city"]]

    if cities_with_special_chars.shape[0] == 0:
        print(f"Cities were successfully converted into english character set")
    else:
        print(f"There was a problem by convert {cities_with_special_chars.shape[0]} cities.")

else:
    print("There are no cities with special characters as letters")


    são paulo, santa barbara d´oeste, santa barbara d´oeste are cities with special characters as letters.
    Convert to english character set... 
    
Cities were successfully converted into english character set


In [44]:
df_sellers["seller_city"].value_counts().sort_values()

seller_city
governador valadares      1
mandaguari                1
sao miguel do oeste       1
coxim                     1
irece                     1
                       ... 
ribeirao preto           52
belo horizonte           68
rio de janeiro           96
curitiba                127
sao paulo               695
Name: count, Length: 609, dtype: int64

In [45]:
#check, if there are cities with special characters

special_characters_list = [",", ";", "/"]


for char in special_characters_list:

    cities_with_special_chars = df_sellers[(df_sellers.seller_city != df_sellers.seller_city.str.replace(char,"")) & (df_sellers.seller_city.isnull() == False)]

    if cities_with_special_chars.shape[0] == 0:
        print(f"There are no cities, which special characters '{char}'\n")

    else:
        print(f"There are {cities_with_special_chars.shape[0]} cities, with special character '{char}':\n")
        print(f"{"\n".join(cities_with_special_chars["seller_city"])}")
        print(f"\n... keep only city before '{char}' ...\n")
        
        df_sellers["seller_city"]= df_sellers["seller_city"].str.split(char).str[0].str.strip()

        if df_sellers[df_sellers["seller_city"] != df_sellers["seller_city"].str.replace(char, "")].shape[0] == 0:
            print(f"Cities with '{char}' succesfully cleaned\n")
        
        else:
            print(f"There was a problem by splitting on '{char}'")

df_sellers["seller_city"] = df_sellers["seller_city"].astype("str")

There are 2 cities, with special character ',':

novo hamburgo, rio grande do sul, brasil
rio de janeiro, rio de janeiro, brasil

... keep only city before ',' ...

Cities with ',' succesfully cleaned

There are no cities, which special characters ';'

There are 15 cities, with special character '/':

auriflama/sp
sao paulo / sao paulo
cariacica / es
sbc/sp
santo andre/sao paulo
sp / sp
maua/sao paulo
mogi das cruzes / sp
barbacena/ minas gerais
rio de janeiro / rio de janeiro
pinhais/pr
ribeirao preto / sao paulo
carapicuiba / sao paulo
sao sebastiao da grama/sp
jacarei / sao paulo

... keep only city before '/' ...

Cities with '/' succesfully cleaned



### seller_state

In [46]:
#check, if there special characters or different writings

seller_state_list = []
uncorrect_counter = 0

for unique in df_sellers["seller_state"].unique():
    
    seller_state_list.append(unique)

seller_state_list.sort()
print(f"The unique seller_states are: {", ".join(seller_state_list)}")


for state in seller_state_list:

    if state != state.lower().strip() or len(state) != 2:

        print(f"The state '{state}' is not in the correct format (2 capital letters)")
        uncorrect_counter += 1
        print("\n... correct the format in seller_state ...\n")
        corrected_state = state.lower().strip()[:2]
        # replace_list.append(state)
        # replace_into_list.append(state.upper().strip()[:2])
        
        df_sellers.seller_state = df_sellers.seller_state.str.replace(state, corrected_state)

        if df_sellers[df_sellers.seller_state == state].shape[0] == 0:
            print(f"Successfully correct the format of '{state}' into '{corrected_state}'")

        else:
            print(f"There was a problem by correct '{state}' into '{corrected_state}'")
    
if uncorrect_counter == 0:    
    print(f"All states are correct formatted")


The unique seller_states are: AC, AM, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RS, SC, SE, SP
The state 'AC' is not in the correct format (2 capital letters)

... correct the format in seller_state ...

Successfully correct the format of 'AC' into 'ac'
The state 'AM' is not in the correct format (2 capital letters)

... correct the format in seller_state ...

Successfully correct the format of 'AM' into 'am'
The state 'BA' is not in the correct format (2 capital letters)

... correct the format in seller_state ...

Successfully correct the format of 'BA' into 'ba'
The state 'CE' is not in the correct format (2 capital letters)

... correct the format in seller_state ...

Successfully correct the format of 'CE' into 'ce'
The state 'DF' is not in the correct format (2 capital letters)

... correct the format in seller_state ...

Successfully correct the format of 'DF' into 'df'
The state 'ES' is not in the correct format (2 capital letters)

... correct the for

### Built a shorter unique key

In [47]:
if seller_id_uniqueness == True:
    
    #starts at 1

    df_sellers["id_sellers"] = np.arange(1, df_sellers.shape[0] + 1)
    

In [48]:
sorted_columns = df_sellers.columns.sort_values()
df_sellers = df_sellers[sorted_columns]
#df_sellers

# Check length of columns

In [49]:
dataframe_dict = {
    "df_sellers": df_sellers,
    "df_products": df_products,
    "df_order_payments": df_order_payments
}

for name, dataframe in dataframe_dict.items():

    columns_list = ";".join(dataframe.columns).split(";")

    for col in columns_list:
        
        if dataframe[col].dtype == "string":

            print(f"""
                    Column '{col}' in {name} 
                     as a max length of: {dataframe[col].astype(str).str.len().max()}
                    has a min length of: {dataframe[col].astype(str).str.len().min()}
                    """)
            
        else:

            print(f"""
                    Column '{col}' in {name} 
                    has a max value of: {dataframe[col].max()}
                    has a min value of: {dataframe[col].min()}
                    """) 
        
       

    print("\n")


                    Column 'id_sellers' in df_sellers 
                    has a max value of: 3095
                    has a min value of: 1
                    

                    Column 'seller_city' in df_sellers 
                     as a max length of: 30
                    has a min length of: 2
                    

                    Column 'seller_id' in df_sellers 
                     as a max length of: 32
                    has a min length of: 32
                    

                    Column 'seller_state' in df_sellers 
                     as a max length of: 2
                    has a min length of: 2
                    

                    Column 'seller_zip_code_prefix' in df_sellers 
                     as a max length of: 5
                    has a min length of: 5
                    



                    Column 'id_products' in df_products 
                    has a max value of: 32950
                    has a min value of: 1
                   

In [50]:
df_sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id_sellers              3095 non-null   int64
 1   seller_city             3095 non-null   str  
 2   seller_id               3095 non-null   str  
 3   seller_state            3095 non-null   str  
 4   seller_zip_code_prefix  3095 non-null   str  
dtypes: int64(1), str(4)
memory usage: 269.8 KB


In [51]:
for name, dataframe in dataframe_dict.items():

    display(dataframe.head(3))

,id_sellers,seller_city,seller_id,seller_state,seller_zip_code_prefix
0,1,campinas,3442f8959a84dea7ee197c632cb2df15,sp,13023
1,2,mogi guacu,d1b65fc7debc3361ea86b5f14c68d2e2,sp,13844
2,3,rio de janeiro,ce3ad9de960102d0677a81f5d0bb7b2d,rj,20031


,id_products,product_category_name,product_description_lenght,product_height_cm,product_id,product_length_cm,product_name_lenght,product_photos_qty,product_weight_g,product_width_cm
0,1,perfumaria,287.0,10.0,1e9e8ef04dbcff4541ed26657ea517e5,16.0,40.0,1.0,225.0,14.0
1,2,artes,276.0,18.0,3aa071139cb16b67ca9e5dea641aaa2f,30.0,44.0,1.0,1000.0,20.0
2,3,esporte_lazer,250.0,9.0,96bd76ec8810374ed1b65e291975717f,18.0,46.0,1.0,154.0,15.0


,id_order_payments,order_id,payment_installments,payment_sequential,payment_type,payment_value
0,1,b81ef226f3fe1789b1e8b2acac839d17,8,1,credit_card,99.33
1,2,a9810da82917af2d9aefd1278f1dcfa0,1,1,credit_card,24.39
2,3,25e8ea4e93396b6fa0d3dd708e76c1bd,1,1,credit_card,65.71


## Export csv local

In [ ]:
df_order_payments.to_csv()